# *preparação de datasets*

In [ ]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from text_features import truncate_text

LABEL_MAP = {
    'gpt': 'openai', 'chatgpt': 'openai', 'openai': 'openai',
    'llama': 'meta', 'meta': 'meta',
    'gemini': 'google', 'gemma': 'google', 'palm': 'google', 'google': 'google',
    'claude': 'anthropic', 'anthropic': 'anthropic',
    'human': 'human', 'wiki': 'human', 'reddit': 'human', 'news': 'human',
}

def map_label(raw):
    raw = str(raw).lower()
    for key, cls in LABEL_MAP.items():
        if key in raw:
            return cls
    return None

## *dataset-exemplo*

In [ ]:
df_stor = pd.read_csv("../datasets/dataset-exemplos.csv", sep=";")

df_stor = df_stor.rename(columns={'Text': 'text', 'Label': 'label'})

df_stor['label'] = df_stor['label'].apply(map_label)

df_stor = df_stor.dropna(subset=['label'])

print('dataset de exemplo:', df_stor.shape)
print(df_stor['label'].value_counts())

## *OpenTuringBench*

In [ ]:
print("A transferir OpenTuringBench...")
dataset_otb = load_dataset("MLNTeam-Unical/OpenTuringBench","in_domain")

dfs_otb = [dataset_otb[split].to_pandas() for split in dataset_otb.keys()]
df_otb = pd.concat(dfs_otb,ignore_index=True)

print("Colunas originais encontradas:", df_otb.columns.tolist())

text_col = next((c for c in df_otb.columns if c.lower() in ['text', 'content', 'generation']), df_otb.columns[0])
label_col = next((c for c in df_otb.columns if c.lower() in ['model', 'label', 'source', 'generator', 'author']), df_otb.columns[1])

df_otb = df_otb.rename(columns={text_col: 'text', label_col: 'label'})

df_otb['label'] = df_otb['label'].apply(map_label)

df_otb = df_otb.dropna(subset=['label'])

print(f"OpenTuringBench processado: {df_otb.shape}")
print(df_otb['label'].value_counts())

## *HC3*

In [ ]:
df_hc3 = pd.read_json("https://huggingface.co/datasets/Hello-SimpleAI/HC3/resolve/main/all.jsonl",lines=True)
print('columns:', df_hc3.columns.tolist())
print('total:', len(df_hc3))

In [ ]:
rows = []
for _,item in df_hc3.iterrows():
    for text in (item.get('human_answers') or []):
        if text and len(text.strip()) >= 50:
            rows.append({'text': text.strip(), 'label': 'human'})
    for text in (item.get('chatgpt_answers') or []):
        if text and len(text.strip()) >= 50:
            rows.append({'text': text.strip(), 'label': 'openai'})

df_hc3 = pd.DataFrame(rows)
print('HC3:', df_hc3.shape)
print(df_hc3['label'].value_counts())

## *Anthropic Persuasion*

In [ ]:
print("A transferir Anthropic/persuasion...")
dataset_persuasion = load_dataset("Anthropic/persuasion", split="train")
df_anthropic = dataset_persuasion.to_pandas()

print("Colunas:", df_anthropic.columns.tolist())
print("Sources:", df_anthropic['source'].value_counts())

df_anthropic = df_anthropic[~df_anthropic['source'].isin(['Human','Control'])]

df_anthropic = df_anthropic.rename(columns={'argument': 'text'})
df_anthropic['label'] = 'anthropic'
df_anthropic = df_anthropic[['text', 'label']]

df_anthropic = df_anthropic.dropna(subset=['text']).drop_duplicates(subset='text')

print(f"Anthropic/persuasion processado: {df_anthropic.shape}")
print(df_anthropic['label'].value_counts())

### juntar, truncar e balancear

In [ ]:
df_all = pd.concat([df_otb, df_anthropic, df_hc3, df_stor], ignore_index=True)

df_all = df_all.dropna(subset=['text', 'label']).drop_duplicates(subset='text')

print('total antes de balancear:', df_all.shape)
print(df_all['label'].value_counts())

In [ ]:
df_all['text'] = df_all['text'].apply(truncate_text)
df_all = df_all.dropna(subset=['text']).reset_index(drop=True)
print('após truncar:', df_all.shape)
print(df_all['text'].str.len().describe().round(1))

In [ ]:
MAX_PER_CLASS = 2000

def balance_class(x):
    if len(x) >= MAX_PER_CLASS:
        return x.sample(n=MAX_PER_CLASS, random_state=42)
    else:
        return x.sample(n=MAX_PER_CLASS, replace=True, random_state=42)

df_balanced = (
    df_all.groupby('label', group_keys=False)
    .apply(balance_class)
    .sample(frac=1, random_state=42)
)

print('após balanceamento:', df_balanced.shape)
print(df_balanced['label'].value_counts())

In [ ]:
df_train, df_temp = train_test_split(df_balanced, test_size=0.3,
                                     stratify=df_balanced['label'], random_state=42)
df_val, df_test   = train_test_split(df_temp, test_size=0.5,
                                     stratify=df_temp['label'], random_state=42)

print(f'treino: {len(df_train)}  validação: {len(df_val)}  teste: {len(df_test)}')

def save_format(df, prefix, path):
    df = df[['text', 'label']].reset_index(drop=True).copy()
    
    df.insert(0, 'ID', [f'{prefix}-{i+1}' for i in range(len(df))])
    df.columns = ['ID', 'Text', 'Label']
    df['Label'] = df['Label'].str.capitalize()
    df.to_csv(path, sep=';', index=False)
    print(f'guardado em: {path}')

save_format(df_train, 'TR', '../datasets/dataset_train.csv')
save_format(df_val,   'VL', '../datasets/dataset_val.csv')
save_format(df_test,  'TE', '../datasets/dataset_test.csv')